# NHC HURDAT2 Download and Parse (Google Colab)

Downloads the latest official **Atlantic HURDAT2 best-track database** from the U.S. National Hurricane Center and converts it into analysis-ready Parquet/CSV files. The Atlantic basin covers the North Atlantic, Caribbean Sea, and Gulf of Mexico.

Outputs include six-hourly and special-time track observations, intensity, pressure, landfall flags, and 34/50/64-knot wind radii for the northeast, southeast, southwest, and northwest quadrants. Wind radii are reported in nautical miles and are generally available beginning in 2004, although individual values may still be missing.

Official sources: [NHC Data Archive](https://www.nhc.noaa.gov/data/) and [HURDAT2 format documentation](https://www.nhc.noaa.gov/data/hurdat/hurdat2-format-nov2019.pdf).

This notebook downloads and parses the storm data. A later spatial-analysis notebook should construct wind-field polygons and intersect them with port gates/geofences. A storm-center track does **not** need to pass through a port county for its wind field to affect the port.


## 1. Install dependencies

Run this cell in Colab. A runtime restart is normally unnecessary.


In [ ]:
%pip install -q polars pyarrow matplotlib


## 2. Configure Google Drive or local output

In Colab, the default output directory is `MyDrive/nhc_hurdat2`. Set `MOUNT_GOOGLE_DRIVE = False` to use temporary Colab storage. Outside Colab, the notebook automatically uses the repository's `data/raw/nhc/hurdat2` and `data/interim/nhc_hurdat2` directories. For automated testing, the environment variable `HURDAT2_BASE_DIR` can override the local destination.


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import re
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import urljoin, urlparse
from urllib.request import Request, urlopen

import matplotlib.pyplot as plt
import polars as pl

IN_COLAB = 'google.colab' in sys.modules
MOUNT_GOOGLE_DRIVE = True
COLAB_DRIVE_DIR = Path('/content/drive/MyDrive/nhc_hurdat2')
COLAB_TEMP_DIR = Path('/content/nhc_hurdat2')
ANALYSIS_START_YEAR = 2013
ANALYSIS_END_YEAR = 2025
FORCE_DOWNLOAD = False

def find_project_root(start: Path | None = None) -> Path:
    candidate = (start or Path.cwd()).resolve()
    for path in (candidate, *candidate.parents):
        if (path / 'pyproject.toml').exists() and (path / 'data').exists():
            return path
    raise FileNotFoundError('Could not find the project root containing pyproject.toml and data/.')

if IN_COLAB and MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = COLAB_DRIVE_DIR
    RAW_DIR = BASE_DIR / 'raw'
    INTERIM_DIR = BASE_DIR / 'processed'
elif IN_COLAB:
    BASE_DIR = COLAB_TEMP_DIR
    RAW_DIR = BASE_DIR / 'raw'
    INTERIM_DIR = BASE_DIR / 'processed'
else:
    override = os.environ.get('HURDAT2_BASE_DIR')
    if override:
        BASE_DIR = Path(override).expanduser().resolve()
        RAW_DIR = BASE_DIR / 'raw'
        INTERIM_DIR = BASE_DIR / 'processed'
    else:
        PROJECT_ROOT = find_project_root()
        BASE_DIR = PROJECT_ROOT / 'data'
        RAW_DIR = BASE_DIR / 'raw/nhc/hurdat2'
        INTERIM_DIR = BASE_DIR / 'interim/nhc_hurdat2'

RAW_DIR.mkdir(parents=True, exist_ok=True)
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
print(f'Running in Colab: {IN_COLAB}')
print(f'Raw directory: {RAW_DIR}')
print(f'Processed directory: {INTERIM_DIR}')


## 3. Discover and download the latest official Atlantic HURDAT2 file

NHC revises HURDAT2 and changes the filename as new seasons and reanalyses are added. The discovery function first reads the official NHC data page, then the official directory index, and selects the candidate with the latest ending year and revision date. A verified February 27, 2026 release is retained only as a fallback if NHC blocks index discovery.


In [ ]:
NHC_DATA_PAGE = 'https://www.nhc.noaa.gov/data/'
NHC_HURDAT_DIR = 'https://www.nhc.noaa.gov/data/hurdat/'
FALLBACK_ATLANTIC_URL = (
    'https://www.nhc.noaa.gov/data/hurdat/'
    'hurdat2-1851-2025-02272026.txt'
)
USER_AGENT = 'supply-chain-resilience/0.1 (academic HURDAT2 downloader)'
ATLANTIC_FILENAME_RE = re.compile(
    r'(hurdat2-(?:atl-)?1851-(\d{4})-(\d{8})\.txt)',
    flags=re.IGNORECASE,
)

def get_bytes(url: str, timeout: int = 180, max_retries: int = 4) -> bytes:
    request = Request(url, headers={'User-Agent': USER_AGENT})
    for attempt in range(max_retries):
        try:
            with urlopen(request, timeout=timeout) as response:
                return response.read()
        except HTTPError as exc:
            if exc.code in {429, 500, 502, 503, 504} and attempt < max_retries - 1:
                wait = 2 ** attempt
                print(f'HTTP {exc.code}; retrying in {wait}s')
                time.sleep(wait)
            else:
                raise
        except (TimeoutError, OSError, URLError) as exc:
            if attempt < max_retries - 1:
                wait = 2 ** attempt
                print(f'Network error; retrying in {wait}s: {exc}')
                time.sleep(wait)
            else:
                raise
    raise RuntimeError('Unreachable')

def discover_latest_atlantic_url() -> tuple[str, str]:
    candidates: list[tuple[int, str, str]] = []
    errors: list[str] = []
    for index_url in (NHC_DATA_PAGE, NHC_HURDAT_DIR):
        try:
            html = get_bytes(index_url).decode('utf-8', errors='replace')
            for filename, end_year, revision in ATLANTIC_FILENAME_RE.findall(html):
                candidates.append((int(end_year), revision, urljoin(NHC_HURDAT_DIR, filename)))
        except Exception as exc:
            errors.append(f'{index_url}: {type(exc).__name__}: {exc}')

    if candidates:
        end_year, revision, url = max(candidates, key=lambda x: (x[0], x[1]))
        return url, f'auto-discovered (through {end_year}, revision {revision})'

    print('Automatic discovery failed; using verified fallback URL.')
    for message in errors:
        print('  ', message)
    return FALLBACK_ATLANTIC_URL, 'verified fallback as of 2026-08-17'

source_url, discovery_method = discover_latest_atlantic_url()
source_filename = Path(urlparse(source_url).path).name
raw_path = RAW_DIR / source_filename

if raw_path.exists() and not FORCE_DOWNLOAD:
    raw_bytes = raw_path.read_bytes()
    print(f'Reusing existing raw file: {raw_path.name}')
else:
    print(f'Downloading: {source_url}')
    raw_bytes = get_bytes(source_url)
    raw_path.write_bytes(raw_bytes)
    print(f'Saved raw file: {raw_path}')

if len(raw_bytes) < 1_000_000 or not raw_bytes.lstrip().startswith(b'AL'):
    raise RuntimeError('Downloaded content does not look like the Atlantic HURDAT2 file.')

retrieved_at_utc = datetime.now(timezone.utc).isoformat()
metadata = {
    'source_url': source_url,
    'source_filename': source_filename,
    'discovery_method': discovery_method,
    'retrieved_at_utc': retrieved_at_utc,
    'n_bytes': len(raw_bytes),
    'sha256': hashlib.sha256(raw_bytes).hexdigest(),
    'official_data_page': NHC_DATA_PAGE,
}
metadata_path = RAW_DIR / f'{raw_path.stem}_metadata.json'
metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
display(metadata)


## 4. Parse HURDAT2

HURDAT2 alternates between storm header rows and observation rows. Missing numeric values are encoded as `-999`. Coordinate suffixes determine the sign: west and south are negative. The parser checks each header's declared observation count and stops on malformed rows instead of silently dropping them.


In [ ]:
RADIUS_COLUMNS = [
    f'r{threshold}_{quadrant}_nm'
    for threshold in (34, 50, 64)
    for quadrant in ('ne', 'se', 'sw', 'nw')
]

def parse_coordinate(value: str) -> float:
    value = value.strip().upper()
    magnitude = float(value[:-1])
    return -magnitude if value[-1] in {'S', 'W'} else magnitude

def parse_optional_int(value: str) -> int | None:
    value = value.strip()
    if not value:
        return None
    parsed = int(value)
    return None if parsed == -999 else parsed

def parse_hurdat2(raw: bytes) -> tuple[pl.DataFrame, list[dict]]:
    lines = raw.decode('utf-8', errors='strict').splitlines()
    rows: list[dict] = []
    header_checks: list[dict] = []
    current: dict | None = None
    observed_for_storm = 0

    def finish_previous() -> None:
        nonlocal observed_for_storm
        if current is None:
            return
        header_checks.append({
            'storm_id': current['storm_id'],
            'declared_entries': current['declared_entries'],
            'parsed_entries': observed_for_storm,
        })
        if observed_for_storm != current['declared_entries']:
            raise ValueError(
                f"{current['storm_id']} declares {current['declared_entries']} entries "
                f'but parser found {observed_for_storm}.'
            )

    for line_number, line in enumerate(lines, start=1):
        if not line.strip():
            continue
        fields = [field.strip() for field in line.split(',')]
        if re.fullmatch(r'[A-Z]{2}\d{6}', fields[0]):
            finish_previous()
            if len(fields) < 3:
                raise ValueError(f'Malformed header at line {line_number}: {line!r}')
            storm_id = fields[0]
            current = {
                'storm_id': storm_id,
                'basin': storm_id[:2],
                'cyclone_number': int(storm_id[2:4]),
                'season': int(storm_id[4:8]),
                'storm_name': fields[1],
                'declared_entries': int(fields[2]),
            }
            observed_for_storm = 0
            continue

        if current is None:
            raise ValueError(f'Observation before first header at line {line_number}.')
        if len(fields) < 20:
            raise ValueError(f'Expected 20 fields at line {line_number}, found {len(fields)}.')

        timestamp = datetime.strptime(fields[0] + fields[1].zfill(4), '%Y%m%d%H%M').replace(
            tzinfo=timezone.utc
        )
        row = {
            **{key: current[key] for key in ('storm_id', 'basin', 'cyclone_number', 'season', 'storm_name')},
            'datetime_utc': timestamp,
            'record_identifier': fields[2] or None,
            'status': fields[3] or None,
            'latitude': parse_coordinate(fields[4]),
            'longitude': parse_coordinate(fields[5]),
            'max_wind_kt': parse_optional_int(fields[6]),
            'min_pressure_mb': parse_optional_int(fields[7]),
            'is_landfall': fields[2] == 'L',
        }
        row.update({name: parse_optional_int(value) for name, value in zip(RADIUS_COLUMNS, fields[8:20])})
        rows.append(row)
        observed_for_storm += 1

    finish_previous()
    tracks = pl.DataFrame(rows, infer_schema_length=None).sort(['storm_id', 'datetime_utc'])
    return tracks, header_checks

tracks, header_checks = parse_hurdat2(raw_bytes)
print(f'Parsed {tracks.height:,} track observations for {tracks["storm_id"].n_unique():,} storms.')
tracks.head()


## 5. Validate and summarize

Zero wind radii are retained because they mean that the threshold wind does not extend into that quadrant. Null radii represent the HURDAT2 `-999` missing code. Do not replace missing radii with zero when constructing exposure fields.


In [ ]:
header_check_df = pl.DataFrame(header_checks)
assert header_check_df.filter(pl.col('declared_entries') != pl.col('parsed_entries')).is_empty()
assert tracks.filter(~pl.col('latitude').is_between(-90, 90)).is_empty()
assert tracks.filter(~pl.col('longitude').is_between(-180, 180)).is_empty()
assert tracks.select(pl.struct(['storm_id', 'datetime_utc']).n_unique()).item() == tracks.height

storm_summary = (
    tracks.group_by(['storm_id', 'basin', 'cyclone_number', 'season', 'storm_name'])
    .agg([
        pl.col('datetime_utc').min().alias('start_time_utc'),
        pl.col('datetime_utc').max().alias('end_time_utc'),
        pl.len().alias('n_track_observations'),
        pl.col('max_wind_kt').max().alias('peak_wind_kt'),
        pl.col('min_pressure_mb').min().alias('minimum_pressure_mb'),
        pl.col('is_landfall').any().alias('has_landfall_record'),
        pl.col('latitude').min().alias('min_latitude'),
        pl.col('latitude').max().alias('max_latitude'),
        pl.col('longitude').min().alias('min_longitude'),
        pl.col('longitude').max().alias('max_longitude'),
    ])
    .sort(['season', 'cyclone_number'])
)

analysis_tracks = tracks.filter(
    pl.col('season').is_between(ANALYSIS_START_YEAR, ANALYSIS_END_YEAR, closed='both')
)
analysis_storms = storm_summary.filter(
    pl.col('season').is_between(ANALYSIS_START_YEAR, ANALYSIS_END_YEAR, closed='both')
)

quality_summary = pl.DataFrame({
    'metric': [
        'track observations', 'storms', 'minimum season', 'maximum season',
        'analysis observations', 'analysis storms', 'observations with any 34kt radius',
    ],
    'value': [
        tracks.height, tracks['storm_id'].n_unique(), tracks['season'].min(), tracks['season'].max(),
        analysis_tracks.height, analysis_tracks['storm_id'].n_unique(),
        tracks.filter(pl.any_horizontal([pl.col(c).is_not_null() for c in RADIUS_COLUMNS[:4]])).height,
    ],
})
display(quality_summary)
display(analysis_storms.tail(20))


## 6. Inspect the Tropical Storm Bill pilot record

This is a parsing and content check only. It does not yet decide whether the wind field intersects the Galveston Bay port geometry.


In [ ]:
bill = tracks.filter(
    (pl.col('season') == 2015)
    & (pl.col('storm_name').str.to_uppercase() == 'BILL')
).sort('datetime_utc')
if bill.is_empty():
    raise RuntimeError('Tropical Storm Bill (2015) was not found; check the source file and parser.')
display(bill)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(bill['longitude'], bill['latitude'], marker='o', linewidth=1.5)
landfall = bill.filter(pl.col('is_landfall'))
if not landfall.is_empty():
    ax.scatter(landfall['longitude'], landfall['latitude'], color='red', s=80, label='HURDAT2 landfall record')
    ax.legend()
ax.set(
    title='Tropical Storm Bill (2015) — HURDAT2 best track',
    xlabel='Longitude', ylabel='Latitude',
)
ax.set_aspect('equal', adjustable='datalim')
plt.show()


## 7. Save analysis-ready files

Parquet preserves UTC timestamps and null wind radii. CSV copies are also written for portability. The official raw text and its SHA-256 metadata remain in the raw directory.


In [ ]:
coverage_start = int(tracks['season'].min())
coverage_end = int(tracks['season'].max())
all_tracks_path = INTERIM_DIR / f'hurdat2_atlantic_tracks_{coverage_start}_{coverage_end}.parquet'
all_storms_path = INTERIM_DIR / f'hurdat2_atlantic_storms_{coverage_start}_{coverage_end}.parquet'
subset_stem = f'hurdat2_atlantic_tracks_{ANALYSIS_START_YEAR}_{ANALYSIS_END_YEAR}'
subset_parquet_path = INTERIM_DIR / f'{subset_stem}.parquet'
subset_csv_path = INTERIM_DIR / f'{subset_stem}.csv'
subset_storms_path = INTERIM_DIR / f'hurdat2_atlantic_storms_{ANALYSIS_START_YEAR}_{ANALYSIS_END_YEAR}.parquet'
subset_storms_csv_path = INTERIM_DIR / f'hurdat2_atlantic_storms_{ANALYSIS_START_YEAR}_{ANALYSIS_END_YEAR}.csv'

tracks.write_parquet(all_tracks_path, compression='zstd')
storm_summary.write_parquet(all_storms_path, compression='zstd')
analysis_tracks.write_parquet(subset_parquet_path, compression='zstd')
analysis_tracks.write_csv(subset_csv_path)
analysis_storms.write_parquet(subset_storms_path, compression='zstd')
analysis_storms.write_csv(subset_storms_csv_path)

output_manifest = pl.DataFrame({
    'file': [
        str(raw_path), str(metadata_path), str(all_tracks_path), str(all_storms_path),
        str(subset_parquet_path), str(subset_csv_path),
        str(subset_storms_path), str(subset_storms_csv_path),
    ],
    'exists': [p.exists() for p in [
        raw_path, metadata_path, all_tracks_path, all_storms_path,
        subset_parquet_path, subset_csv_path, subset_storms_path, subset_storms_csv_path,
    ]],
    'bytes': [p.stat().st_size for p in [
        raw_path, metadata_path, all_tracks_path, all_storms_path,
        subset_parquet_path, subset_csv_path, subset_storms_path, subset_storms_csv_path,
    ]],
})
display(output_manifest)
print('Done. The 2013–2025 track Parquet is the main input for port-exposure construction:')
print(subset_parquet_path)


## Field notes for the next exposure notebook

- `record_identifier == 'L'` marks a HURDAT2 center landfall; blank identifiers are normal.
- `status` includes tropical depression (`TD`), tropical storm (`TS`), hurricane (`HU`), extratropical (`EX`), subtropical states, and other classifications documented by NHC.
- `r34_*_nm`, `r50_*_nm`, and `r64_*_nm` are quadrant radii in nautical miles. They are not circular buffers and should be converted to asymmetric geodesic polygons.
- `0` means no wind of that threshold in the quadrant; null means unavailable. Preserve this distinction.
- HURDAT2 is a best-track/post-analysis dataset, appropriate for retrospective exposure measurement rather than real-time forecasting.
- Interpolate storm position and radii cautiously between observations when constructing hourly exposure. Retain original observations and mark interpolated rows.
